# FSR Scraping Pipeline — Step-by-Step Dev Test

**Safe for dev testing** — writes only to `__dev_tmp` table, never touches real tables or volumes (read-only).

**Cluster:** `ai_dev_dbr` or Serverless (AI Dev workspace)

Stages:
1. PDF field extraction (pdfplumber — first page)
2. LLM normalization (15-field canonical schema)
3. IBAT + Event Vision enrichment (read-only joins)
4. Write to `__dev_tmp` table

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
%pip install --quiet pdfplumber litellm

In [ ]:
# ── Cell 2: Restart Python (run after pip install) ───────────────────────────
dbutils.library.restartPython()

In [ ]:
# ── Cell 3: Imports + Configuration ──────────────────────────────────────────
import os, json, time, re, warnings, base64
from pathlib import Path
from urllib.parse import quote

import requests
import urllib3
import pdfplumber
import pandas as pd

warnings.filterwarnings("ignore")
urllib3.disable_warnings()

# ── LLM Configuration ──
# Secret scope is still blocked — using direct key for now.
# Paste your API key when prompted by the widget, or set it here directly.
try:
    LITELLM_API_KEY = dbutils.widgets.get("LITELLM_API_KEY")
except Exception:
    LITELLM_API_KEY = ""

LITELLM_BASE_URL = "https://dev-gateway.apps.gevernova.net"
LLM_MODEL = "gemini-3-flash"

# ── Volume paths (read-only) ──
PDF_VOLUME_PATHS = [
    "/Volumes/viud/ing_ud_fieldvision/fv_field_service_report",
    "/Volumes/viud/ing_ud_fsr_manual/manual_field_service_report/ecrt_reports",
]

# ── Output: dev temp table ONLY ──
DEV_OUTPUT_TABLE = "main.gp_services_sdg_poc.fsr_scraped_file_mapping_ref__dev_tmp"

print(f"LLM base URL:   {LITELLM_BASE_URL}")
print(f"LLM model:      {LLM_MODEL}")
print(f"API key set:    {bool(LITELLM_API_KEY)}")
print(f"Output table:   {DEV_OUTPUT_TABLE}")
print(f"Volumes:        {len(PDF_VOLUME_PATHS)}")

## Stage 1: PDF Field Extraction (pdfplumber)
Read a single test PDF and extract key:value fields from the first page.

In [ ]:
# ── Cell 4: List a few PDFs from the first volume ────────────────────────────
# Read-only — just listing file names via Files API

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

def _get_dbr_auth():
    ws_url = "https://gevernova-ai-dev-dbr.cloud.databricks.com"
    try:
        token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
        return ws_url, token
    except Exception:
        return ws_url, os.getenv("DATABRICKS_TOKEN", "")

def list_volume_files(volume_path, max_files=10):
    ws_url, token = _get_dbr_auth()
    encoded = quote(volume_path.lstrip("/"), safe="")
    url = f"{ws_url}/api/2.0/fs/directories/{encoded}"
    headers = {"Authorization": f"Bearer {token}"}
    resp = requests.get(url, headers=headers, timeout=60, verify=False)
    resp.raise_for_status()
    files = []
    for entry in resp.json().get("contents", []):
        if not entry.get("is_directory"):
            path = entry.get("path", f"{volume_path.rstrip('/')}/{entry.get('name', '')}")
            if path.startswith("dbfs:"):
                path = path[5:]
            files.append(path)
            if len(files) >= max_files:
                break
    return files

sample_pdfs = list_volume_files(PDF_VOLUME_PATHS[0], max_files=5)
print(f"Found {len(sample_pdfs)} sample files:")
for p in sample_pdfs:
    print(f"  {p}")

In [ ]:
# ── Cell 5: Extract fields from a single PDF (Stage 1) ───────────────────────

def extract_fields_from_pdf(pdf_path: str) -> dict:
    """Stage 1: Extract key:value fields and title from the first page."""
    fields = {"PDF Name / path / identifier": pdf_path}
    try:
        with pdfplumber.open(pdf_path) as pdf:
            first_page = pdf.pages[0]
            text = first_page.extract_text()
            if not text:
                return fields
            lines = text.splitlines()
            current_key = None
            # Title = lines before first colon-delimited field
            title_lines = []
            for line in lines:
                if ":" in line:
                    break
                title_lines.append(line.strip())
            if title_lines:
                fields["Title"] = " ".join(title_lines)
            # key:value pairs
            for line in lines:
                if ":" in line:
                    parts = line.split(":", 1)
                    key = parts[0].strip()
                    value = parts[1].strip()
                    current_key = key
                    if key in fields:
                        fields[key] = f"{fields[key]} | {value}"
                    else:
                        fields[key] = value
                else:
                    if current_key:
                        fields[current_key] += " " + line.strip()
    except Exception as e:
        print(f"  [WARN] Error extracting {pdf_path}: {e}")
    return fields

# Test on first PDF
test_pdf = sample_pdfs[0]
stage1_result = extract_fields_from_pdf(test_pdf)
print(f"PDF: {test_pdf}")
print(f"Fields extracted: {len(stage1_result)}")
print(json.dumps(stage1_result, indent=2))

In [ ]:
# ── Cell 6: Stage 1 on a batch of PDFs ───────────────────────────────────────
# Extract fields from all sample PDFs
stage1_batch = [extract_fields_from_pdf(p) for p in sample_pdfs]
print(f"Extracted fields from {len(stage1_batch)} PDFs")
for i, rec in enumerate(stage1_batch):
    title = rec.get("Title", "[no title]")[:60]
    print(f"  [{i+1}] {title} — {len(rec)} fields")

## Stage 2: LLM Normalization

Send the extracted fields to the LLM gateway for normalization into the 15-field canonical schema.

**Requires:** working `dev-gateway.apps.gevernova.net` + API key.

In [ ]:
# ── Cell 7: LLM call helper + normalization prompt ───────────────────────────

SYSTEM_PROMPT = (
    "You are an expert in structuring technical data. "
    "Your task is to process the provided input json and output a structured/normalized "
    "JSON format according to user instructions. Focus on clarity, completeness, and "
    "following the JSON schema provided. Do not include any internal reasoning or system "
    "details in the output. Ensure all responses are fact-based, and safe. "
    "Follow responsible AI principles without over-restricting harmless tasks."
)

NORMALIZATION_PROMPT = """Your task is to process JSON input and output a normalized JSON format
with the following fields only:

ESN, Equipment Sys ID, Equipment Type, Equipment Class / Code, Event Type,
EV Project ID, EV Equipment Event ID, OFS Event ID, FSP project ID, xxx project id,
PDF Name / path / identifier, FSR Number (#), Report Issued Date, Outage Start Date, Outage End Date.

If the PDF field contains an Oracle Project Id, map it to OFS Event ID only. But do not include field values that start with "EV" and "EVP" under OFS Event ID.
Map the PDF field containing "EV-" to EV Equipment Event ID only.
Map the PDF field containing "EVP-" to EV Project ID only.
Map the PDF field containing "SY" to Equipment Sys ID only.
If the PDF field contains a Field Service Project Id or "FSP-", map it to FSP project ID only.
If the PDF field contains a project id that starts with other prefixes (A-, C-, etc.), map it to xxx project id only.

**Strictly adhere to the following instructions while extracting and normalizing data**:
Do not miss any information that is present in the input and try to be as much accurate as possible in retrieving the values for the above fields.
**When extracting data, if a record contains multiple distinct values across ESN and Equipment Sys ID fields, split the record into separate rows by pairing values positionally (first with first, second with second, etc.), while duplicating all other field values unchanged (except for Equipment Type and Equipment Class / Code). Set the Equipment Type and Equipment Class / Code field values to empty strings in the split rows. Do not split or omit any parts for other field values even if multiple distinct values are present.**
If no exact match is found for a field, see if you can infer it from similar labels or context.
If no relevant information is found, output it as an empty string.
Consider as many records as provided in the input batch. Do not omit any records.
Do not include any reasoning or commentary, only valid JSON output.
"""


def call_llm(prompt: str) -> str:
    """Call LLM via direct HTTP. Retries with exponential backoff."""
    base = LITELLM_BASE_URL.rstrip("/")
    candidate_urls = [f"{base}/chat/completions", f"{base}/v1/chat/completions"]
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
    }
    headers = {
        "Authorization": f"Bearer {LITELLM_API_KEY}",
        "Content-Type": "application/json",
    }

    for attempt in range(1, 4):
        for url in candidate_urls:
            try:
                resp = requests.post(url, headers=headers, json=payload, timeout=120, verify=False)
                if resp.status_code == 404:
                    continue
                if resp.status_code >= 400:
                    raise RuntimeError(f"HTTP {resp.status_code}: {resp.text[:300]}")
                return resp.json()["choices"][0]["message"]["content"]
            except Exception as e:
                if "404" in str(e):
                    continue
                if attempt == 3:
                    raise
                wait = 5 * (2 ** (attempt - 1))
                print(f"  [WARN] Attempt {attempt}/3 failed ({e}); retrying in {wait}s")
                time.sleep(wait)
    raise RuntimeError("All LLM call attempts failed")


def strip_json_fences(text: str) -> str:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()


print("LLM helpers defined ✓")

In [ ]:
# ── Cell 8: Run Stage 2 on the batch ─────────────────────────────────────────
# Sends all stage1 records to LLM in one batch for normalization

assert LITELLM_API_KEY, "Set LITELLM_API_KEY widget before running this cell"

prompt = (
    f"Here is a JSON list of PDF fields:\n{json.dumps(stage1_batch, indent=2)}\n\n"
    + NORMALIZATION_PROMPT
)

print(f"Sending {len(stage1_batch)} records to LLM ({LLM_MODEL})...")
t0 = time.time()
raw_response = call_llm(prompt)
elapsed = time.time() - t0
print(f"LLM responded in {elapsed:.1f}s")

cleaned = strip_json_fences(raw_response)
stage2_results = json.loads(cleaned)
if isinstance(stage2_results, dict) and "FSR_data" in stage2_results:
    stage2_results = stage2_results["FSR_data"]

print(f"Normalized {len(stage2_results)} rows")
print(json.dumps(stage2_results[:2], indent=2))

## Stage 3: IBAT + Event Vision Enrichment

Read-only joins against IBAT equipment master and Event Vision SOT to fill in missing equipment_type, equipment_code, event dates, and project IDs.

In [ ]:
# ── Cell 9: Stage 3 — IBAT + Event Vision enrichment ─────────────────────────
from pyspark.sql.functions import col, when, trim, concat, lit, upper, regexp_replace
from pyspark.sql.types import StringType

COLUMN_RENAME_MAP = {
    "ESN": "esn",
    "Equipment Sys ID": "equipment_sys_id",
    "Equipment Type": "equipment_type",
    "Equipment Class / Code": "equipment_code",
    "Event Type": "event_type",
    "EV Project ID": "ev_project_id",
    "EV Equipment Event ID": "ev_equipment_event_id",
    "OFS Event ID": "ofs_event_id",
    "FSP project ID": "fsp_project_id",
    "xxx project id": "xxx_project_id",
    "PDF Name / path / identifier": "pdf_name",
    "FSR Number (#)": "fsr_number",
    "Report Issued Date": "report_issued_date",
    "Outage Start Date": "outage_start_date",
    "Outage End Date": "outage_end_date",
}
FINAL_COLS = list(COLUMN_RENAME_MAP.values())

# Load lookup tables (read-only)
enrichment_enabled = True
try:
    ibat = spark.read.table("vgpd.prm_std_views.ibat_equipment_mst").select(
        upper(trim(col("equipment_sys_id"))).cast(StringType()).alias("ibat_equipment_sys_id"),
        upper(trim(col("equip_serial_number"))).cast(StringType()).alias("ibat_equip_serial_number"),
        col("equipment_type").cast(StringType()).alias("ibat_equipment_type"),
        col("equipment_sub_class").cast(StringType()).alias("ibat_equipment_code"),
    )
    ev_sot = spark.read.table("vgpd.fsr_std_views.eventmgmt_event_vision_sot").select(
        col("ev_project_id").cast(StringType()).alias("sot_ev_project_id"),
        col("ev_equipment_event_id").cast(StringType()).alias("sot_ev_equipment_event_id"),
        col("ev_gtm_id").cast(StringType()).alias("sot_ev_gtm_id"),
        col("fsp_project_id").cast(StringType()).alias("sot_fsp_project_id"),
        col("ev_event_type").cast(StringType()).alias("sot_event_type"),
        col("p6_outage_start_date").cast(StringType()).alias("sot_outage_start_date"),
        col("p6_outage_end_date").cast(StringType()).alias("sot_outage_end_date"),
    )
    # Force metadata resolution
    ibat.limit(1).count()
    ev_sot.limit(1).count()
    print(f"IBAT loaded ✓  |  EV SOT loaded ✓")
except Exception as e:
    print(f"[WARN] Lookup table access failed: {e}")
    print("       Continuing without Stage 3 enrichment — will use Stage 2 output directly")
    enrichment_enabled = False

In [ ]:
# ── Cell 10: Run Stage 3 enrichment ──────────────────────────────────────────

def enrich_batch(normalised_records, ibat_df, ev_sot_df):
    """IBAT + Event Vision enrichment on a batch of normalized records."""
    pdf = pd.DataFrame(normalised_records).rename(columns=COLUMN_RENAME_MAP)
    for c in FINAL_COLS:
        if c not in pdf.columns:
            pdf[c] = ""

    fsr_df = spark.createDataFrame(pdf)
    fsr_df = fsr_df.select([col(c).cast(StringType()).alias(c) for c in pdf.columns])

    fsr_df = (
        fsr_df
        .withColumn("ev_project_id", trim(col("ev_project_id")))
        .withColumn("ev_equipment_event_id", trim(col("ev_equipment_event_id")))
        .withColumn("ofs_event_id", trim(col("ofs_event_id")))
        .withColumn("fsp_project_id", trim(col("fsp_project_id")))
        .withColumn("fsp_project_id_stripped", regexp_replace(col("fsp_project_id"), "FSP-", ""))
        .withColumn("ev_project_id_stripped", regexp_replace(col("ev_project_id"), "EVP-", ""))
        .withColumn("ev_equipment_event_id_stripped", regexp_replace(col("ev_equipment_event_id"), "EV-", ""))
    )

    enriched = fsr_df.join(
        ibat_df,
        (fsr_df["equipment_sys_id"] == ibat_df["ibat_equipment_sys_id"])
        | (fsr_df["esn"] == ibat_df["ibat_equip_serial_number"]),
        "left",
    )
    enriched = (
        enriched
        .withColumn("esn", when((col("esn").isNull()) | (col("esn") == ""), col("ibat_equip_serial_number")).otherwise(col("esn")))
        .withColumn("equipment_sys_id", when((col("equipment_sys_id").isNull()) | (col("equipment_sys_id") == ""), col("ibat_equipment_sys_id")).otherwise(col("equipment_sys_id")))
        .withColumn("equipment_type", when((col("equipment_type").isNull()) | (col("equipment_type") == ""), col("ibat_equipment_type")).otherwise(col("equipment_type")))
        .withColumn("equipment_code", when((col("equipment_code").isNull()) | (col("equipment_code") == ""), col("ibat_equipment_code")).otherwise(col("equipment_code")))
    )

    enriched = enriched.join(
        ev_sot_df,
        (enriched["ev_project_id_stripped"] == ev_sot_df["sot_ev_project_id"])
        | (enriched["ev_equipment_event_id_stripped"] == ev_sot_df["sot_ev_equipment_event_id"])
        | (enriched["ofs_event_id"] == ev_sot_df["sot_ev_gtm_id"])
        | (enriched["fsp_project_id_stripped"] == ev_sot_df["sot_fsp_project_id"]),
        "left",
    )
    enriched = (
        enriched
        .withColumn("ev_project_id", when((col("ev_project_id").isNull()) | (col("ev_project_id") == ""), concat(lit("EVP-"), col("sot_ev_project_id"))).otherwise(col("ev_project_id")))
        .withColumn("ev_equipment_event_id", when((col("ev_equipment_event_id").isNull()) | (col("ev_equipment_event_id") == ""), concat(lit("EV-"), col("sot_ev_equipment_event_id"))).otherwise(col("ev_equipment_event_id")))
        .withColumn("ofs_event_id", when((col("ofs_event_id").isNull()) | (col("ofs_event_id") == ""), col("sot_ev_gtm_id")).otherwise(col("ofs_event_id")))
        .withColumn("fsp_project_id", when((col("fsp_project_id").isNull()) | (col("fsp_project_id") == ""), concat(lit("FSP-"), col("sot_fsp_project_id"))).otherwise(col("fsp_project_id")))
        .withColumn("event_type", when((col("event_type").isNull()) | (col("event_type") == ""), col("sot_event_type")).otherwise(col("event_type")))
        .withColumn("outage_start_date", when((col("outage_start_date").isNull()) | (col("outage_start_date") == ""), col("sot_outage_start_date")).otherwise(col("outage_start_date")))
        .withColumn("outage_end_date", when((col("outage_end_date").isNull()) | (col("outage_end_date") == ""), col("sot_outage_end_date")).otherwise(col("outage_end_date")))
    )
    return enriched.select([col(c) for c in FINAL_COLS])


def rows_to_fallback_df(normalised_records):
    """Fallback when lookup tables are inaccessible: write normalized rows directly."""
    pdf = pd.DataFrame(normalised_records).rename(columns=COLUMN_RENAME_MAP)
    for c in FINAL_COLS:
        if c not in pdf.columns:
            pdf[c] = ""
    return spark.createDataFrame(pdf).select([col(c).cast(StringType()).alias(c) for c in FINAL_COLS])


# Run enrichment
if enrichment_enabled:
    enriched_df = enrich_batch(stage2_results, ibat, ev_sot)
    print("Stage 3 enrichment complete ✓")
else:
    enriched_df = rows_to_fallback_df(stage2_results)
    print("Stage 3 skipped (no lookup access) — using Stage 2 output directly")

print(f"Result rows: {enriched_df.count()}")
enriched_df.show(truncate=False)

## Stage 4: Write to Dev Temp Table

Writes enriched results to `__dev_tmp` table only. **Never touches the real `fsr_scraped_file_mapping_ref`.**

In [ ]:
# ── Cell 11: Write to dev temp table ─────────────────────────────────────────
enriched_df.write.mode("overwrite").saveAsTable(DEV_OUTPUT_TABLE)

result_df = spark.read.table(DEV_OUTPUT_TABLE)
print(f"Wrote {result_df.count()} rows to {DEV_OUTPUT_TABLE}")
print(f"Distinct PDFs: {result_df.select('pdf_name').distinct().count()}")

# Null rate summary
print("\nNull/empty rates:")
total = result_df.count()
for c in FINAL_COLS:
    null_count = result_df.filter(col(c).isNull() | (col(c) == "")).count()
    pct = 100 * null_count / max(total, 1)
    indicator = "⚠️" if pct > 50 else "✓"
    print(f"  {indicator} {c}: {null_count}/{total} ({pct:.0f}%)")

display(result_df)

In [ ]:
# ── Cell 12: Cleanup — drop temp table when done ─────────────────────────────
# Uncomment and run when you're done testing:

# spark.sql(f"DROP TABLE IF EXISTS {DEV_OUTPUT_TABLE}")
# print(f"Dropped {DEV_OUTPUT_TABLE}")